[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/en_clase/S10_02_mlp_keras_en_clase.ipynb)

# S10 · 02 — Tu primer MLP en Keras: cancelaciones y precio por noche · versión en clase

⏱ 60 min en clase · Sprint 10 · UD 4.2

**Contexto TurisData.** Con `reservas_turisdata.csv` ya has predicho cancelaciones con modelos clásicos. Ahora usarás un **perceptrón multicapa (MLP)** de Keras para dos tareas:

1. **Clasificación:** ¿se cancelará la reserva? (`cancelada`).
2. **Regresión:** ¿cuál es el `precio_noche`?

**Este cuaderno tiene 3 ejercicios** (1 a 3), los que se hacen en el aula. El MLP de regresión (precio por noche), su evaluación frente a una referencia, el mini-reto de la red ancha y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1, 2 y 3 son los 3 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # menos mensajes de TensorFlow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, mean_absolute_error
from sklearn.linear_model import LogisticRegression, Ridge

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)   # fija NumPy, Python y TensorFlow a la vez
print("TensorFlow", tf.__version__)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar las reservas y crear mes_llegada

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas reservas y columnas hay y qué proporción se cancela?
2. ¿Qué porcentaje acierta un modelo que diga siempre «no cancela»?
3. ¿Por qué se crea `mes_llegada` en lugar de usar la fecha en bruto?

In [ ]:
df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
print(df.shape, "· tasa de cancelación:", round(df["cancelada"].mean(), 3))
df.head(3)

### ✏️ A3 · EJERCICIO 1 de 3 — Columnas numéricas y categóricas

**Tu turno:** Define dos listas: `cols_num` (las columnas numéricas: `antelacion_dias`, `noches`, `adultos`, `ninos`, `peticiones_especiales`, `cliente_repetidor`, `cancelaciones_previas`, `mes_llegada`) y `cols_cat` (las categóricas: `tipo_habitacion`, `regimen`, `canal`, `pais_origen`, `tarifa`, `deposito`). Para la clasificación añadiremos `precio_noche` a las numéricas más abajo.

**Debe salir:** ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A3).

1. ¿Por qué no se incluye `cancelada` entre las columnas de entrada?
2. ¿Por qué no se usa `id_reserva`?
3. ¿En qué se diferencia el tratamiento de las numéricas y de las categóricas?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A3, **después de intentarlo**.

In [ ]:
# pista: dos listas de nombres de columna, sin id_reserva, sin fechas y sin el objetivo
cols_num = ...
cols_cat = ...

assert len(cols_num) == 8 and len(cols_cat) == 6, "Deben ser 8 numéricas y 6 categóricas"
assert all(c in df.columns for c in cols_num + cols_cat), "Alguna columna no existe: revisa los nombres"
assert "cancelada" not in cols_num + cols_cat and "id_reserva" not in cols_num, "No incluyas el objetivo ni el identificador"
print("✔ Ejercicio 1 correcto")

### 👀 A4 · EJEMPLO — Separar entrenamiento, validación y prueba y preparar las matrices

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿Cuántas reservas hay en entrenamiento, validación y prueba?
2. ¿Por qué el escalador se ajusta solo con el entrenamiento?
3. ¿Para qué sirve el conjunto de validación y por qué la prueba se toca una sola vez?

In [ ]:
def preparar(objetivo, extra_num=()):
    """Devuelve matrices ya escaladas/codificadas y el preprocesador ajustado."""
    numericas = list(cols_num) + list(extra_num)
    X = df[numericas + cols_cat]
    y = df[objetivo].to_numpy()
    estrato = y if objetivo == "cancelada" else None
    X_res, X_pru, y_res, y_pru = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=estrato)
    estrato = y_res if objetivo == "cancelada" else None
    X_ent, X_val, y_ent, y_val = train_test_split(X_res, y_res, test_size=0.2, random_state=SEMILLA, stratify=estrato)
    pre = ColumnTransformer([("num", StandardScaler(), numericas),
                             ("cat", OneHotEncoder(handle_unknown="ignore"), cols_cat)])
    A_ent = pre.fit_transform(X_ent).astype("float32")     # ajusta SOLO con entrenamiento
    A_val = pre.transform(X_val).astype("float32")
    A_pru = pre.transform(X_pru).astype("float32")
    return A_ent, A_val, A_pru, y_ent, y_val, y_pru, pre

# Clasificación: el precio es un dato conocido al reservar, así que puede ser una variable de entrada
Xc_ent, Xc_val, Xc_pru, yc_ent, yc_val, yc_pru, pre_c = preparar("cancelada", extra_num=["precio_noche"])
print("Clasificación:", Xc_ent.shape, Xc_val.shape, Xc_pru.shape)

### ✏️ A5 · EJERCICIO 2 de 3 — Construir el MLP de clasificación

**Tu turno:** Escribe `construir_mlp_clasificacion(n_entradas, capas=(32, 16))` que devuelva un modelo `keras.Sequential` con una capa `Input`, una capa `Dense` con `relu` por cada número de `capas`, y una salida `Dense(1, activation="sigmoid")`. Debe **compilarlo** con el optimizador `adam`, la pérdida `binary_crossentropy` y las métricas `AUC` (llamada `auc`) y `accuracy`.

**Debe salir:** el resumen del modelo (3 capas `Dense`, **1 697** parámetros en total) y ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuántos parámetros tiene el modelo y cómo se calculan los de la primera capa?
2. ¿Por qué la capa de salida es `Dense(1, activation="sigmoid")`?
3. ¿Qué hace `compile` y qué pérdida usa aquí?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: recorre `capas` añadiendo Dense(unidades, activation="relu"); termina con Dense(1, "sigmoid") y compila
def construir_mlp_clasificacion(n_entradas, capas=(32, 16)):
    ...

mlp_c = construir_mlp_clasificacion(Xc_ent.shape[1])
mlp_c.summary()
assert len(mlp_c.layers) == 3, "Con capas=(32, 16) debe haber 3 capas Dense (2 ocultas + salida)"
assert mlp_c.layers[-1].activation.__name__ == "sigmoid", "La salida debe ser sigmoid"
assert mlp_c.count_params() == Xc_ent.shape[1] * 32 + 32 + 32 * 16 + 16 + 16 + 1, "Número de parámetros inesperado: revisa capas y tamaños"
print("✔ Ejercicio 2 correcto")

### 👀 A6 · EJEMPLO — Entrenar con fit y ver las curvas de aprendizaje

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A6).

1. ¿En qué época tiene la validación su menor pérdida y cuánto vale?
2. ¿Qué indica que la pérdida de entrenamiento siga bajando mientras la de validación sube?
3. ¿Qué diferencia hay entre el AUC de entrenamiento y el de validación al final?

In [ ]:
hist_c = mlp_c.fit(Xc_ent, yc_ent, validation_data=(Xc_val, yc_val), epochs=30, batch_size=64, verbose=0)

def dibujar_curvas(hist, metrica, titulo):
    h = hist.history
    fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
    ejes[0].plot(h["loss"], label="entrenamiento"); ejes[0].plot(h["val_loss"], label="validación")
    ejes[0].set_title("Pérdida"); ejes[0].set_xlabel("época"); ejes[0].set_ylabel("pérdida"); ejes[0].legend()
    ejes[1].plot(h[metrica], label="entrenamiento"); ejes[1].plot(h["val_" + metrica], label="validación")
    ejes[1].set_title(metrica.upper()); ejes[1].set_xlabel("época"); ejes[1].set_ylabel(metrica); ejes[1].legend()
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

dibujar_curvas(hist_c, "auc", "MLP de cancelaciones: curvas de aprendizaje")

### 👀 A7 · EJEMPLO — Evaluar en prueba y comparar con una referencia

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué AUC tienen la regresión logística y el MLP?
2. ¿Qué exactitud logra el modelo que siempre dice «no cancela»?
3. ¿Qué conclusión sacas para TurisData?

In [ ]:
prob_mlp = mlp_c.predict(Xc_pru, verbose=0).ravel()
logistica = LogisticRegression(max_iter=1000).fit(Xc_ent, yc_ent)
prob_log = logistica.predict_proba(Xc_pru)[:, 1]

tabla = pd.DataFrame({
    "modelo": ["Siempre 'no cancela'", "Regresión logística", "MLP (32-16)"],
    "AUC": [0.5, roc_auc_score(yc_pru, prob_log), roc_auc_score(yc_pru, prob_mlp)],
    "exactitud": [1 - yc_pru.mean(), accuracy_score(yc_pru, prob_log > 0.5), accuracy_score(yc_pru, prob_mlp > 0.5)],
}).round(3)
tabla

### ✏️ A8 · EJERCICIO 3 de 3 — ¿Gana el MLP a la regresión logística?

**Tu turno:** Guarda en `auc_mlp` el AUC del MLP en prueba y en `auc_log` el de la regresión logística (a partir de `prob_mlp` y `prob_log`, con `roc_auc_score`). Después guarda en `mlp_gana` un booleano (`True`/`False`) que diga si el MLP supera a la regresión logística **por más de 0.02 de AUC**. Esta última respuesta puede ser `False`: no siempre gana la red.

**Debe salir:** `AUC MLP = 0.727 · AUC logística = 0.746 · ¿gana la red por más de 0.02? False` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A8).

1. ¿Qué valen `auc_mlp`, `auc_log` y `mlp_gana`?
2. ¿Por qué se usa un margen de 0,02 y no solo `auc_mlp > auc_log`?
3. ¿Por qué se usan probabilidades y no las clases para el AUC?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A8, **después de intentarlo**.

In [ ]:
# pista: roc_auc_score(yc_pru, probabilidades) y una comparación con margen 0.02
auc_mlp = ...
auc_log = ...
mlp_gana = ...

assert 0.6 < auc_mlp < 0.9 and 0.6 < auc_log < 0.9, "Los AUC deberían estar entre 0.6 y 0.9. ¿Has usado las probabilidades y no las clases?"
assert isinstance(mlp_gana, bool), "mlp_gana debe ser True o False"
print(f"AUC MLP = {auc_mlp:.3f} · AUC logística = {auc_log:.3f} · ¿gana la red por más de 0.02? {mlp_gana}")
print("✔ Ejercicio 3 correcto")

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/para_casa/S10_02_mlp_keras.ipynb)** trae el MLP de regresión del precio por noche, su evaluación frente a una referencia, el mini-reto de la red ancha y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga las herramientas: `numpy`, `pandas`, `matplotlib`, `tensorflow` y `keras` (la librería de redes neuronales) y, de `scikit-learn`, la separación de datos, el preprocesado de columnas, las métricas y los dos modelos de referencia (`LogisticRegression` y `Ridge`).
    - `os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"` silencia los mensajes técnicos de TensorFlow.
    - `keras.utils.set_random_seed(SEMILLA)` fija las semillas de NumPy, Python y TensorFlow a la vez para que los resultados se repitan. Importar TensorFlow tarda unos segundos.
- **Qué debe salir:**
    - `TensorFlow x.y.z` (la versión instalada, por ejemplo 2.21.0) y `Listo ✔`.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.
    - `ModuleNotFoundError: No module named 'tensorflow'` → fuera de Colab, instala TensorFlow en tu entorno.

## 1. Preparar los datos para una red neuronal

#### A2 · Ejemplo — Cargar las reservas y crear mes_llegada

**Contexto TurisData.** Con `reservas_turisdata.csv` ya has predicho cancelaciones con modelos clásicos. Ahora usarás un **perceptrón multicapa (MLP)** de Keras para dos tareas: **clasificación** (¿se cancelará la reserva?) y **regresión** (¿cuál es el `precio_noche`?).

**Idea.** Las redes son sensibles a la escala: si `antelacion_dias` va de 0 a 400 y `adultos` de 1 a 4,
la primera "grita" más que la segunda. Por eso:
- las variables **numéricas** se **estandarizan** (media 0, desviación 1);
- las **categóricas** se convierten en columnas 0/1 (*one-hot*; ojo: `pais_origen` tiene 138 reservas sin país y el codificador las cuenta como una categoría más);
- el escalador se **ajusta solo con el entrenamiento** (si no, "espiaríamos" los datos de prueba: fuga de datos).

Además crearemos `mes_llegada` a partir de la fecha (la temporada influye) y **no** usaremos
`id_reserva` ni las fechas en bruto.

- **Qué hace:**
    - `pd.read_csv(..., parse_dates=["fecha_llegada"])` lee las reservas y convierte la fecha de llegada en fecha de verdad.
    - `df["mes_llegada"] = df["fecha_llegada"].dt.month` crea una columna con el mes (la temporada influye).
    - El `print` muestra la forma de la tabla y la proporción de canceladas (`df["cancelada"].mean()`); `df.head(3)` enseña las tres primeras filas.
- **Qué debe salir:**
    - `(6000, 19) · tasa de cancelación: 0.459` y una tabla con las 3 primeras reservas (19 columnas, entre ellas `id_reserva`, `antelacion_dias`, `canal`, `tarifa`, `precio_noche`, `cancelada` y la nueva `mes_llegada`).
- **Cómo leerlo:**
    - Hay 6 000 reservas y el 45,9 % se cancela: las dos clases están bastante equilibradas.
- **Si falla:**
    - `NameError: name 'dato' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas reservas y columnas hay y qué proporción se cancela?* → **6 000** reservas, **19** columnas (con `mes_llegada`) y se cancela el **45,9 %**.
2. *¿Qué porcentaje acierta un modelo que diga siempre «no cancela»?* → El **54,1 %** (1 − 0,459): es la referencia de «no saber nada».
3. *¿Por qué se crea `mes_llegada` en lugar de usar la fecha en bruto?* → Porque la red necesita **números con significado**: el mes (1 a 12) recoge la temporada, mientras que una fecha completa no se puede usar tal cual.

</details>

#### A3 · Ejercicio 1 — Columnas numéricas y categóricas

- **Qué hace:**
    - `cols_num` es la lista de las 8 columnas numéricas que se estandarizarán y `cols_cat` la de las 6 categóricas que se codificarán en 0/1.
    - No incluyen `id_reserva` (un identificador), las fechas en bruto ni `cancelada` (el objetivo).
- **Qué debe salir:**
    - `✔ Ejercicio 1 correcto`.
- **Si falla:**
    - `Deben ser 8 numéricas y 6 categóricas` → te falta alguna columna o sobra alguna.
    - `Alguna columna no existe: revisa los nombres` → hay un error de escritura en un nombre (por ejemplo `noche` en vez de `noches`).
    - `No incluyas el objetivo ni el identificador` → has dejado `cancelada` o `id_reserva`.
    - `NameError: name 'df' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Por qué no se incluye `cancelada` entre las columnas de entrada?* → Porque es **lo que queremos predecir**: si la red la viera como entrada, haría trampa.
2. *¿Por qué no se usa `id_reserva`?* → Porque es solo un número de identificación: **no explica nada** del comportamiento y la red podría usarlo para memorizar.
3. *¿En qué se diferencia el tratamiento de las numéricas y de las categóricas?* → Las numéricas se **estandarizan** (media 0, desviación 1) y las categóricas se pasan a columnas 0/1 (**one-hot**).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
cols_num = ["antelacion_dias", "noches", "adultos", "ninos", "peticiones_especiales",
            "cliente_repetidor", "cancelaciones_previas", "mes_llegada"]
cols_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A4 · Ejemplo — Separar entrenamiento, validación y prueba y preparar las matrices

- **Entrenamiento (64 %)**: donde la red ajusta sus pesos.
- **Validación (16 %)**: para vigilar durante el entrenamiento y tomar decisiones (cuántas épocas, qué arquitectura).
- **Prueba (20 %)**: se toca **una sola vez**, al final, para dar la cifra honesta.

- **Qué hace:**
    - `preparar(objetivo, extra_num)` es una función que deja los datos listos para una red. Separa el 20 % para **prueba** y, del resto, otro 20 % para **validación** (64 % / 16 % / 20 %); en clasificación reparte las clases proporcionalmente (`stratify`).
    - El `ColumnTransformer` estandariza las numéricas (`StandardScaler`) y hace *one-hot* de las categóricas (`OneHotEncoder`); se **ajusta solo con el entrenamiento** (`fit_transform`) y se aplica a validación y prueba (`transform`).
    - La llamada para clasificar añade `precio_noche` como variable de entrada (se conoce al reservar). Devuelve `Xc_ent`, `Xc_val`, `Xc_pru`, sus respuestas (`yc_...`) y el preprocesador.
- **Qué debe salir:**
    - `Clasificación: (3840, 35) (960, 35) (1200, 35)`.
- **Cómo leerlo:**
    - 3 840 + 960 + 1 200 = 6 000 reservas. Cada reserva tiene ahora **35 columnas** (9 numéricas más las columnas 0/1 de las categóricas).
- **Si falla:**
    - `NameError: name 'cols_num' is not defined` → falta el ejercicio 1.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas reservas hay en entrenamiento, validación y prueba?* → **3 840**, **960** y **1 200**.
2. *¿Por qué el escalador se ajusta solo con el entrenamiento?* → Para **no usar información de validación ni de prueba**: si la media y la desviación incluyeran esos datos, «espiaríamos» el examen (fuga de datos) y la evaluación sería demasiado optimista.
3. *¿Para qué sirve el conjunto de validación y por qué la prueba se toca una sola vez?* → La **validación** sirve para vigilar y decidir (épocas, arquitectura) durante el entrenamiento; la **prueba** se guarda para dar al final la cifra honesta.

</details>

## 2. Un MLP para clasificar cancelaciones

#### A5 · Ejercicio 2 — Construir el MLP de clasificación

**Idea.** Un MLP es una pila de capas `Dense` (todas las neuronas de una capa conectadas con todas las de la siguiente).
Para clasificación binaria:
- **Capas ocultas:** `relu`.
- **Capa de salida:** 1 neurona con `sigmoid` (probabilidad de cancelar).
- **Pérdida:** `binary_crossentropy` (la misma que programaste a mano).
- **Métricas:** `AUC` y `accuracy`. Recuerda que la referencia de "no saber nada" es acertar ≈ 54 % (no cancela).

- **Qué hace:**
    - `keras.Sequential([keras.layers.Input(shape=(n_entradas,))])` crea un modelo que apila capas y que espera `n_entradas` columnas.
    - El bucle añade una capa `Dense(unidades, activation="relu")` por cada número de `capas` (aquí 32 y 16 neuronas) y después la salida `Dense(1, activation="sigmoid")`.
    - `compile(...)` fija el optimizador (`adam`, que ajusta los pesos), la pérdida (`binary_crossentropy`, la misma que programaste a mano) y las métricas (`AUC` con nombre `auc` y `accuracy`).
    - La comprobación construye el modelo con las 35 columnas de entrada, muestra su `summary()` y verifica el número de capas, la activación de salida y el número de parámetros.
- **Qué debe salir:**
    - Una tabla `Model: "sequential"` con tres capas: `dense` (32 neuronas, 1 152 parámetros), `dense_1` (16, 528) y `dense_2` (1, 17), con `Total params: 1,697`, y después `✔ Ejercicio 2 correcto`.
- **Cómo leerlo:**
    - Los parámetros de cada capa son (entradas × neuronas) + neuronas: 35·32 + 32 = 1 152; 32·16 + 16 = 528; 16·1 + 1 = 17.
- **Si falla:**
    - `Con capas=(32, 16) debe haber 3 capas Dense (2 ocultas + salida)` → falta la capa de salida o te sobra una capa.
    - `La salida debe ser sigmoid` → la última capa no lleva `activation="sigmoid"`.
    - `Número de parámetros inesperado: revisa capas y tamaños` → las capas ocultas no tienen 32 y 16 neuronas, o no se usan los `capas` recibidos.
    - `AttributeError: 'NoneType' object has no attribute 'summary'` → la función no devuelve el modelo (falta `return modelo`).
    - `NameError: name 'Xc_ent' is not defined` → falta el ejemplo de preparar los datos.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántos parámetros tiene el modelo y cómo se calculan los de la primera capa?* → **1 697**. La primera capa: 35 entradas × 32 neuronas + 32 sesgos = **1 152**.
2. *¿Por qué la capa de salida es `Dense(1, activation="sigmoid")`?* → Porque es una clasificación binaria: **una** neurona cuya salida (0 a 1) se interpreta como la probabilidad de cancelar.
3. *¿Qué hace `compile` y qué pérdida usa aquí?* → Prepara el modelo para entrenar: elige el optimizador (`adam`), la **pérdida** (`binary_crossentropy`) y las métricas a vigilar (`AUC` y `accuracy`).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
def construir_mlp_clasificacion(n_entradas, capas=(32, 16)):
    modelo = keras.Sequential([keras.layers.Input(shape=(n_entradas,))])
    for unidades in capas:
        modelo.add(keras.layers.Dense(unidades, activation="relu"))
    modelo.add(keras.layers.Dense(1, activation="sigmoid"))
    modelo.compile(optimizer="adam", loss="binary_crossentropy",
                   metrics=[keras.metrics.AUC(name="auc"), "accuracy"])
    return modelo
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A6 · Ejemplo — Entrenar con fit y ver las curvas de aprendizaje

- **época**: una pasada completa por los datos de entrenamiento;
- **lote (*batch*)**: cuántas filas se usan antes de actualizar los pesos (aquí 64);
- `validation_data`: datos para vigilar en cada época **sin entrenar con ellos**.

- **Qué hace:**
    - `mlp_c.fit(...)` entrena 30 épocas con lotes de 64 filas; `validation_data` evalúa en cada época con la validación **sin entrenar con ella**; `verbose=0` no escribe nada. Devuelve el historial (`hist_c`).
    - `dibujar_curvas(hist, metrica, titulo)` es una función que dibuja dos gráficos: la pérdida (izquierda) y la métrica (derecha) de entrenamiento y validación por época.
- **Qué debe salir:**
    - Dos gráficos. Izquierda: la pérdida de entrenamiento baja de ≈ 0,69 a ≈ 0,52; la de validación baja hasta ≈ 0,59 hacia la época 6 y luego sube hasta ≈ 0,62. Derecha: el AUC de entrenamiento sube hasta ≈ 0,82 y el de validación se queda en ≈ 0,74.
- **Cómo leerlo:**
    - **Lee las curvas.** La pérdida de entrenamiento sigue bajando, pero la de validación se estanca y luego sube: la red empieza a **memorizar** el entrenamiento. Es sobreajuste. En la siguiente práctica lo controlaremos.
- **Si falla:**
    - `NameError: name 'mlp_c' is not defined` → falta el ejercicio 2. Tarda unos segundos en entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿En qué época tiene la validación su menor pérdida y cuánto vale?* → Hacia la **época 6**, con ≈ **0,59**; después sube.
2. *¿Qué indica que la pérdida de entrenamiento siga bajando mientras la de validación sube?* → **Sobreajuste**: la red memoriza el entrenamiento y deja de generalizar.
3. *¿Qué diferencia hay entre el AUC de entrenamiento y el de validación al final?* → ≈ **0,82** frente a ≈ **0,74**: la red lo hace mejor con los datos que ya ha visto.

</details>

#### A7 · Ejemplo — Evaluar en prueba y comparar con una referencia

Una cifra sola no dice nada: hay que compararla con un modelo sencillo. Usaremos una **regresión logística**.

- **Qué hace:**
    - `mlp_c.predict(Xc_pru)` da la probabilidad de cancelar de las 1 200 reservas de prueba (`prob_mlp`).
    - `LogisticRegression(...).fit(...)` entrena la referencia y `predict_proba(...)[:, 1]` da sus probabilidades (`prob_log`).
    - `tabla` compara tres modelos con el AUC y la exactitud en prueba: «siempre no cancela» (AUC 0,5 y exactitud = proporción de no canceladas), la regresión logística y el MLP.
- **Qué debe salir:**
    - Una tabla: «Siempre 'no cancela'» → AUC 0,500 y exactitud 0,541; «Regresión logística» → 0,746 y 0,678; «MLP (32-16)» → 0,727 y 0,661.
- **Cómo leerlo:**
    - La red **no mejora** a la regresión logística: la supera en nada y queda 0,02 por debajo en AUC. Una red más compleja no siempre gana en datos tabulares.
- **Si falla:**
    - `NameError: name 'mlp_c' is not defined` → falta el ejercicio 2 (y el entrenamiento del ejemplo anterior).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué AUC tienen la regresión logística y el MLP?* → **0,746** y **0,727**.
2. *¿Qué exactitud logra el modelo que siempre dice «no cancela»?* → **0,541**: es la referencia de «no saber nada».
3. *¿Qué conclusión sacas para TurisData?* → Que el MLP **no aporta** frente a la regresión logística, que es más simple, rápida y explicable. Había que comprobarlo antes de celebrar la red.

</details>

#### A8 · Ejercicio 3 — ¿Gana el MLP a la regresión logística?

- **Qué hace:**
    - `roc_auc_score(yc_pru, prob_mlp)` calcula el AUC del MLP en prueba y `roc_auc_score(yc_pru, prob_log)` el de la logística (hay que pasar las **probabilidades**, no las clases).
    - `bool(auc_mlp - auc_log > 0.02)` es `True` solo si el MLP supera a la logística por más de 0,02.
- **Qué debe salir:**
    - `AUC MLP = 0.727 · AUC logística = 0.746 · ¿gana la red por más de 0.02? False` y `✔ Ejercicio 3 correcto`.
- **Si falla:**
    - `Los AUC deberían estar entre 0.6 y 0.9. ¿Has usado las probabilidades y no las clases?` → has pasado clases (0/1) o los datos equivocados a `roc_auc_score`.
    - `mlp_gana debe ser True o False` → `mlp_gana` no es un booleano (por ejemplo, has guardado la diferencia).
    - `NameError: name 'prob_mlp' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué valen `auc_mlp`, `auc_log` y `mlp_gana`?* → **0,727**, **0,746** y **`False`**.
2. *¿Por qué se usa un margen de 0,02 y no solo `auc_mlp > auc_log`?* → Porque diferencias muy pequeñas son **ruido**: no basta con ganar por una milésima para justificar un modelo más complejo.
3. *¿Por qué se usan probabilidades y no las clases para el AUC?* → Porque el AUC mide cómo **ordena** el modelo las reservas por riesgo; con solo 0/1 se pierde esa información.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
auc_mlp = roc_auc_score(yc_pru, prob_mlp)
auc_log = roc_auc_score(yc_pru, prob_log)
mlp_gana = bool(auc_mlp - auc_log > 0.02)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>